In [2]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd

OUT_DIR = Path("../outputs")
MODEL_FILE = OUT_DIR / "best_churn_model.joblib"
CARD_FILE = OUT_DIR / "model_card.json"            # optional
INPUT_CSV = Path("../Dataset/Telco-Customer-Churn.csv")
OUTPUT_CSV = OUT_DIR / "scores_demo.csv"
HIGH, MEDIUM = 0.40, 0.20     # band cut-offs (you will tune HIGH in Step 5)
TOP_REASONS = 3



def clean(df: pd.DataFrame) -> pd.DataFrame:
    """Same cleaning as training: blank TotalCharges for tenure 0 becomes 0."""
    df = df.copy()
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
    df.loc[(df["tenure"] == 0) & df["TotalCharges"].isnull(), "TotalCharges"] = 0.0
    return df


def band(p: float) -> str:
    return "High" if p >= HIGH else "Medium" if p >= MEDIUM else "Low"


def pretty(feature: str, scaled_value: float) -> str:
    """'cat__Contract_Month-to-month' -> 'Contract = Month-to-month'."""
    kind, name = feature.split("__", 1)
    if kind == "num":
        return f"{name} ({'high' if scaled_value > 0 else 'low'})"
    return name.replace("_", " = ", 1)


def score_customers(df: pd.DataFrame, bundle: dict, card: dict) -> pd.DataFrame:
    pre, model = bundle["preprocess"], bundle["model"]

    # 1. Clean and check the columns
    df = clean(df)
    expected = list(pre.feature_names_in_)
    needs_senior_fix = "SeniorCitizen" in expected
    missing = [c for c in expected if c not in df.columns]
    if missing:
        raise ValueError(f"Input file is missing required column(s): {missing}")

    # 2. Build the feature table exactly as in training
    X = df[expected].copy()
    if needs_senior_fix and pd.api.types.is_numeric_dtype(X["SeniorCitizen"]):
        X["SeniorCitizen"] = X["SeniorCitizen"].map({0: "No", 1: "Yes"})

    # 3. Transform and predict
    Xt = pre.transform(X)
    Xt = Xt.toarray() if hasattr(Xt, "toarray") else np.asarray(Xt)
    prob = model.predict_proba(Xt)[:, 1]

    out = pd.DataFrame({
        "customerID": df["customerID"].values,
        "churn_probability": prob.round(4),
        "risk_band": [band(p) for p in prob],
    })

    # 4. Reasons: the features pushing each customer's risk UP the most
    #    (only available for linear models such as Logistic Regression)
    if hasattr(model, "coef_"):
        names = pre.get_feature_names_out()
        contrib = Xt * model.coef_[0]
        order = np.argsort(-contrib, axis=1)[:, :TOP_REASONS]
        for k in range(TOP_REASONS):
            out[f"reason_{k + 1}"] = [
                pretty(names[row[k]], Xt[i, row[k]]) if contrib[i, row[k]] > 0 else ""
                for i, row in enumerate(order)
            ]

    # 5. Traceability columns
    out["model"] = bundle["name"]
    out["model_version"] = card.get("version", "unknown")
    out["scored_at"] = pd.Timestamp.now().strftime("%Y-%m-%d")

    # Keep the true outcome if the file has one (useful for monitoring later)
    if "Churn" in df.columns:
        out["actual_churn"] = df["Churn"].values
    return out


def run(input_csv=INPUT_CSV, output_csv=OUTPUT_CSV):
    bundle = joblib.load(MODEL_FILE)
    card = json.loads(CARD_FILE.read_text()) if CARD_FILE.exists() else {}
    scores = score_customers(pd.read_csv(input_csv), bundle, card)
    Path(output_csv).parent.mkdir(parents=True, exist_ok=True)
    scores.to_csv(output_csv, index=False)
    print(f"Saved {len(scores):,} scores to {output_csv}")
    print(scores["risk_band"].value_counts())
    return scores


scores = run()
scores.head(10)

Saved 7,043 scores to ..\outputs\scores_demo.csv
risk_band
High      3445
Low       2397
Medium    1201
Name: count, dtype: int64


,customerID,churn_probability,risk_band,reason_1,reason_2,reason_3,model,model_version,scored_at,actual_churn
0,7590-VHVEG,0.8048,High,tenure (low),MonthlyCharges (low),Contract = Month-to-month,Logistic Regression,unknown,2026-10-12,No
1,5575-GNVDE,0.1181,Low,MonthlyCharges (low),TechSupport = No,DeviceProtection = Yes,Logistic Regression,unknown,2026-10-12,No
2,3668-QPYBK,0.5321,High,tenure (low),Contract = Month-to-month,MonthlyCharges (low),Logistic Regression,unknown,2026-10-12,Yes
3,7795-CFOCW,0.0810,Low,MonthlyCharges (low),DeviceProtection = Yes,OnlineBackup = No,Logistic Regression,unknown,2026-10-12,No
4,9237-HQITU,0.8545,High,tenure (low),InternetService = Fiber optic,Contract = Month-to-month,Logistic Regression,unknown,2026-10-12,Yes
5,9305-CDSKC,0.9090,High,tenure (low),InternetService = Fiber optic,Contract = Month-to-month,Logistic Regression,unknown,2026-10-12,Yes
6,1452-KIOVK,0.7199,High,InternetService = Fiber optic,Contract = Month-to-month,tenure (low),Logistic Regression,unknown,2026-10-12,No
7,6713-OKOMC,0.5237,High,tenure (low),MonthlyCharges (low),Contract = Month-to-month,Logistic Regression,unknown,2026-10-12,No
8,7892-POOKP,0.8166,High,InternetService = Fiber optic,Contract = Month-to-month,StreamingMovies = Yes,Logistic Regression,unknown,2026-10-12,Yes
9,6388-TABGU,0.0336,Low,TotalCharges (high),MonthlyCharges (low),TechSupport = No,Logistic Regression,unknown,2026-10-12,No


In [3]:
# 1. Row count matches the input file
print(len(scores) == len(pd.read_csv(INPUT_CSV)))     # expect True

# 2. Probabilities are valid
print(scores["churn_probability"].between(0, 1).all())  # expect True

# 3. Scores match your earlier test-set predictions
old = pd.read_csv(OUT_DIR / "churn_predictions.csv")
cmp = old.merge(scores, on="customerID")
print(len(cmp), (cmp["churn_probability_x"] - cmp["churn_probability_y"]).abs().max())
# expect 1,409 rows and a difference of 0.0

True
True
1409 0.0
